# Vectors & dot products: hands-on (optional)
**LLMs, Animated · Lesson P1.1**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## 1 · The dot product, by hand
$$a \cdot b = \sum_i a_i b_i$$
Implement it with a plain Python loop (no `np.dot`), so you see there's nothing hidden.

In [ ]:
def dot(a, b):
    assert len(a) == len(b), "vectors must have the same length"
    # TODO: multiply matching entries and add them up
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
assert dot([3, 1], [1, 2]) == 5
assert dot([2, -1], [1, 2]) == 0          # perpendicular
assert dot([1, 2, 2], [2, 1, 2]) == 8
x, y = np.random.default_rng(0).normal(size=(2, 100))
assert np.isclose(dot(x, y), x @ y)       # matches NumPy's built-in
print("✓ exercise 1")

## 2 · Length and cosine similarity
$$\lVert a\rVert = \sqrt{a \cdot a} \qquad \cos\theta = \frac{a \cdot b}{\lVert a\rVert\,\lVert b\rVert}$$
Implement `norm` (using your `dot`) and `cosine_similarity`.

In [ ]:
def norm(a):
    # TODO
    raise NotImplementedError("Exercise 2a")

def cosine_similarity(a, b):
    # TODO
    raise NotImplementedError("Exercise 2b")

**Check 2**

In [ ]:
assert np.isclose(norm([3, 4]), 5)
assert np.isclose(norm([3, 1]), np.sqrt(10))
assert np.isclose(cosine_similarity([3, 1], [6, 2]), 1.0)      # same direction, different length
assert np.isclose(cosine_similarity([2, -1], [1, 2]), 0.0)     # perpendicular
assert np.isclose(cosine_similarity([1, 0], [-5, 0]), -1.0)    # opposite
assert np.isclose(cosine_similarity([4, 3], [3, 4]), 0.96)     # the lesson's faded example
print("✓ exercise 2")

## 3 · Two formulas, one number
For 2-D vectors you can measure each arrow's direction with `np.arctan2(y, x)`. Compute the angle $\theta$ between $a = (3, 1)$ and $b = (1, 2)$ that way (no dot products allowed), then check that $\lVert a\rVert\,\lVert b\rVert\cos\theta$ equals $a \cdot b$.

In [ ]:
a, b = np.array([3.0, 1.0]), np.array([1.0, 2.0])
theta = None           # TODO: angle between a and b, in radians, from arctan2 only
geometric = None       # TODO: |a| |b| cos(theta)

**Check 3**

In [ ]:
assert theta is not None and geometric is not None
assert np.isclose(np.degrees(theta), 45.0)
assert np.isclose(geometric, dot(a, b))
print(f"✓ exercise 3:  θ = {np.degrees(theta):.1f}°,  |a||b|cos θ = {geometric:.3f} = a·b")

## 4 · Rank toy "embeddings" by similarity
Here are hand-made 4-dimensional "embeddings" (real ones are learned and have hundreds or thousands of dimensions). Imagine the dimensions mean roughly *animal-ness, cuteness, vehicle-ness, size*.

Fill in `ranked`: the other words sorted from **most** to **least** cosine-similar to `"cat"`.

In [ ]:
emb = {
    "cat":     np.array([0.9, 0.8, 0.0, 0.2]),
    "kitten":  np.array([0.8, 1.0, 0.0, 0.1]),
    "dog":     np.array([0.9, 0.6, 0.0, 0.4]),
    "tiger":   np.array([0.9, 0.2, 0.0, 0.8]),
    "truck":   np.array([0.0, 0.0, 1.0, 0.9]),
    "scooter": np.array([0.0, 0.3, 0.9, 0.2]),
}


In [ ]:
ranked = None   # TODO: list of the other 5 words, most similar to "cat" first

**Check 4**

In [ ]:
assert ranked is not None and len(ranked) == 5 and "cat" not in ranked
assert ranked[0] == "kitten" and ranked[-1] == "truck", ranked
for w in ranked:
    print(f"  cat · {w:8s} cos = {cosine_similarity(emb['cat'], emb[w]):+.3f}")
print("✓ exercise 4")

## 5 · Random directions in high dimensions are nearly perpendicular
The lesson claims that the cosine similarity of two **random** directions in $d$ dimensions is typically about $\pm 1/\sqrt{d}$. Test it: for each `d`, draw 2,000 random pairs (`rng.normal(size=d)`) and store the **standard deviation** of their cosine similarities in `spread[d]`. Use NumPy for speed: `x @ y` and `np.linalg.norm`.

In [ ]:
rng = np.random.default_rng(1)
dims = [2, 10, 100, 1000, 4096]
spread = {}
# TODO: for each d, fill spread[d] with the std of cosine similarity over 2000 random pairs

**Check 5**

In [ ]:
assert set(spread) == set(dims)
for d in dims[1:]:
    assert abs(spread[d] * np.sqrt(d) - 1) < 0.15, (d, spread[d])
for d in dims:
    print(f"  d = {d:5d}:  std of cos = {spread[d]:.4f}   (1/√d = {1/np.sqrt(d):.4f})")
print("✓ exercise 5: in 4,096 dimensions, a cosine of 0.3 is far from random")

## Experiment · all-pairs similarity in one matrix multiply
Normalise every embedding to length 1, stack them as rows of a matrix `E`, and `E @ E.T` gives the cosine similarity of **every pair at once**: each entry is a dot product (lesson P1.2 shows why). This is exactly how vector search and attention batch their work on a GPU.

In [ ]:
words = list(emb)
E = np.stack([emb[w] / np.linalg.norm(emb[w]) for w in words])
S = E @ E.T
plt.figure(figsize=(5, 4))
plt.imshow(S, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(words)), words, rotation=45); plt.yticks(range(len(words)), words)
plt.colorbar(label="cosine similarity"); plt.title("E @ E.T"); plt.tight_layout(); plt.show()

## Recap
- The dot product is a multiply-and-add loop; $a\cdot b = \lVert a\rVert\,\lVert b\rVert\cos\theta$.
- Cosine similarity keeps only direction: it's how embeddings are compared.
- Random high-dimensional directions are nearly perpendicular (cosine ≈ ±1/√d), so even moderate similarities carry signal.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def dot(a, b):
    assert len(a) == len(b), "vectors must have the same length"
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def norm(a):
    return float(np.sqrt(dot(a, a)))

def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
a, b = np.array([3.0, 1.0]), np.array([1.0, 2.0])
theta = np.arctan2(b[1], b[0]) - np.arctan2(a[1], a[0])
geometric = norm(a) * norm(b) * np.cos(theta)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
ranked = sorted((w for w in emb if w != "cat"),
                key=lambda w: cosine_similarity(emb["cat"], emb[w]), reverse=True)

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
rng = np.random.default_rng(1)
dims = [2, 10, 100, 1000, 4096]
spread = {}
for d in dims:
    x = rng.normal(size=(2000, d))
    y = rng.normal(size=(2000, d))
    cos = (x * y).sum(axis=1) / (np.linalg.norm(x, axis=1) * np.linalg.norm(y, axis=1))
    spread[d] = float(cos.std())